# SIBA Jittor 逐模块测试

本 Notebook 读取PyTorch生成的同一输入、同一权重和同一步训练基准，再逐项检查Jittor结果。测试结论按实际误差输出，不把功能通过写成严格数值相等。


## 1. 环境与路径


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path('/root/autodl-tmp/SIBA-Jittor')
JITTOR_PYTHON = Path('/root/autodl-tmp/envs/JittorDome/bin/python')
OFFICIAL_CHECKPOINT = PROJECT_ROOT / 'official_pytorch/checkpoint/SIBA_epoch60.pth'
TEST_ROOT = PROJECT_ROOT / 'logs/demo_module_tests'
PYTORCH_REFERENCE = TEST_ROOT / 'pytorch_seed2025.npz'
JITTOR_REPORT = TEST_ROOT / 'jittor_seed2025.json'
TEST_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
!"{JITTOR_PYTHON}" -c "import jittor as jt; jt.flags.use_cuda=1; print('Jittor', jt.__version__); print('CUDA', jt.flags.use_cuda)"


## 2. 运行全部Jittor检查


In [ ]:
if not PYTORCH_REFERENCE.exists():
    raise FileNotFoundError('请先运行 SIBA_PyTorch_逐模块测试.ipynb 生成PyTorch基准。')
!"{JITTOR_PYTHON}" "{PROJECT_ROOT / 'tools/check_jittor_alignment.py'}" --project-root "{PROJECT_ROOT}" --checkpoint "{OFFICIAL_CHECKPOINT}" --reference "{PYTORCH_REFERENCE}" --output "{JITTOR_REPORT}" --use-cuda


In [ ]:
report = json.loads(JITTOR_REPORT.read_text())
checks = report['checks']

def show_checks(names):
    rows = []
    for name in names:
        rows.append({'name': name, **checks[name]})
    return pd.DataFrame(rows)

def show_prefix(prefix):
    return show_checks([name for name in checks if name.startswith(prefix)])

print('检查项数量:', len(checks))
print('Jittor版本:', report['jittor_version'])


## 3. 数据读取与配对


In [ ]:
print((PROJECT_ROOT / 'logs/alignment/data_loader_report.json').read_text())
print((PROJECT_ROOT / 'logs/alignment/training_dataset_validation.json').read_text())
print((PROJECT_ROOT / 'logs/alignment/test_dataset_pairing.json').read_text())


## 4. 权重键与初始参数


In [ ]:
print('checkpoint参数键:', report['checkpoint_parameter_keys'])
print('Jittor模型参数键:', report['model_parameter_keys'])
initial = show_prefix('parameter_initial__')
display(initial.sort_values('max_abs', ascending=False).head(10))
print('全部初始参数最大绝对误差:', initial['max_abs'].max())


## 5. SE与Res-SE特征提取


In [ ]:
display(show_checks(['activation__ir_conv', 'activation__vi_conv']))


## 6. LayerNorm、归一化与自注意力


In [ ]:
display(show_checks(['activation__ir_sa_0', 'activation__vi_sa_0']))


## 7. CBSM源图权重


In [ ]:
display(show_checks([
    'activation__weight_ir', 'activation__weight_irI',
    'activation__weight_vi', 'activation__weight_viI',
]))


## 8. 四路交叉注意力


In [ ]:
display(show_checks([
    'activation__ir2vi_ca_0', 'activation__irI2vi_ca_0',
    'activation__vi2ir_ca_0', 'activation__viI2ir_ca_0',
]))


## 9. 特征拼接与最终输出


In [ ]:
display(show_checks([
    'activation__mixed', 'activation__fuse_conv',
    'activation__output', 'manual_vs_model_output',
]))


## 10. Laplacian、Intensity与Sobel损失


In [ ]:
display(show_checks(['loss_laplacian', 'loss_intensity', 'loss_sobel', 'loss_total']))


## 11. 原生Jittor反向传播


In [ ]:
display(pd.DataFrame([{'stage': 'gradient_preclip', **report['aggregates']['gradient_preclip']}]))
print('相对L2误差约1.05%，不满足严格1e-3阈值。')


## 12. PyTorch语义梯度裁剪


In [ ]:
display(show_checks(['clip_total_norm']))
display(pd.DataFrame([
    {'stage': 'native_gradient_postclip', **report['aggregates']['gradient_postclip']},
    {'stage': 'reference_gradient_postclip', **report['aggregates']['reference_gradient_postclip']},
]))


## 13. PyTorch兼容Adam


In [ ]:
display(pd.DataFrame([
    {'stage': 'native_parameter_update', **report['aggregates']['parameter_update']},
    {'stage': 'reference_gradient_parameter_after_step', **report['aggregates']['reference_gradient_parameter_after_step']},
]))
print('相同参考梯度下，裁剪和Adam实现通过；原生一步更新不宣称严格相等。')


## 14. 分层测试结论


In [ ]:
display(pd.DataFrame([report['summary']]).T.rename(columns={0: 'value'}))


## 15. 官方权重下706张输出对齐


In [ ]:
RUN_TAG = '20260727_siba_official_protocol'
rows = []
for dataset in ['MSRS', 'M3FD_2x', 'TNO']:
    result = json.loads((PROJECT_ROOT / f'results/output_alignment_{RUN_TAG}/{dataset}/summary.json').read_text())
    rows.append({
        'dataset': dataset,
        'images': result['compared_images'],
        'max_abs_uint8': result['global_max_abs_uint8'],
        'mean_abs_uint8': result['global_mean_abs_uint8'],
        'filenames_shapes_match': result['all_filenames_and_shapes_match'],
    })
display(pd.DataFrame(rows))


## 16. 完整60轮训练


In [ ]:
display(Image(filename=str(PROJECT_ROOT / f'results/training_analysis_{RUN_TAG}/loss_curve.png')))
print((PROJECT_ROOT / f'results/training_analysis_{RUN_TAG}/training_log_summary.json').read_text())


## 17. 指标与性能


In [ ]:
display(pd.read_csv(PROJECT_ROOT / f'results/metrics_{RUN_TAG}/metrics_summary.csv'))
display(pd.read_csv(PROJECT_ROOT / f'results/performance_summary_{RUN_TAG}/inference_timing.csv'))
print((PROJECT_ROOT / f'results/performance_summary_{RUN_TAG}/gpu_monitor_summary.json').read_text())


## 18. 复现边界


In [ ]:
print('推理：官方权重下高度一致。')
print('训练：Jittor完成全量60轮并收敛。')
print('严格训练步等价：未通过，不能表述为逐步完全相同。')
print('RoadScene 200对名单：作者未公开，不能表述为训练集完全相同。')
